## Classification and Evaluation

I evaluated whether a company's four preceding recorded earnings events could aid in predicting whether its next reported earnings would beat analyst's estimates. To do this I compared logistic regression, decision trees, random forests and XGBoost against simple baseline predictors.

I used chronological validation to evaluate performance on later earnings announcements. I also examined how well the models identified meet-or-miss outcomes, which were less common than earnings beats in the dataset.

### Chronological Validation

I used four expanding-window validation folds covering 2020 through 2023. For each fold, I trained the models on observations from earlier years and evaluated them on the following year.

I used chronological splits because randomly dividing the earnings records could allow later observations to influence predictions for earlier announcements. The 2024 observations were reserved for the final holdout evaluation and were excluded from these experiments.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

split_path = RESULTS_DIR / "split_coverage.json"

with split_path.open(encoding="utf-8") as file:
    split_results = json.load(file)


fold_rows = []

for fold in split_results["folds"]:
    fold_rows.append(
        {
            "Validation Year": fold["validation_year"],
            "Training Samples": fold["training_samples"],
            "Validation Samples": fold["validation_samples"],
        }
    )

fold_table = pd.DataFrame(fold_rows)

display(fold_table)

print(f"Development samples: {split_results['development_samples']:,}")
print(f"Reserved 2024 samples: {split_results['holdout']['samples']:,}")

,Validation Year,Training Samples,Validation Samples
0,2020,24591,13449
1,2021,38040,13840
2,2022,51880,14809
3,2023,66689,15664


Development samples: 82,353
Reserved 2024 samples: 7,996


#### Validation Setup

The number of training observations increased with each validation year, from 24,591 for the 2020 fold to 66,689 for the 2023 fold. Each model was therefore trained using the historical observations available before its evaluation year. I used the same validation folds for the baseline predictors and all four classifiers to make their results comparable.

### Baseline Predictors

Before evaluating the machine learning models, I tested four baseline predictors:

1. **Majority Class:** Predicts whichever class was most common in the training data.
2. **Always Beat:** Predicts an earnings beat for every observation.
3. **Previous Outcome:** Repeats the company's most recent recorded earnings outcome.
4. **Historical Beat Rate:** Uses the beat history from the four preceding recorded events to estimate the likelihood of another beat.

I compared their performance across the same four chronological validation folds used for the classifiers. I used macro F1 as the primary evaluation metric because it gives equal weight to both outcome classes.

In [2]:
baseline_path = RESULTS_DIR / "baseline_results.json"

with baseline_path.open(encoding="utf-8") as file:
    baseline_results = json.load(file)


baseline_names = {
    "majority_class": "Majority Class",
    "always_beat": "Always Beat",
    "previous_outcome": "Previous Outcome",
    "historical_beat_rate": "Historical Beat Rate",
}

baseline_rows = []

for fold in baseline_results["folds"]:
    for name, label in baseline_names.items():
        scores = fold["models"][name]

        baseline_rows.append(
            {
                "Baseline": label,
                "Year": fold["validation_year"],
                "Accuracy": scores["accuracy"],
                "Macro F1": scores["macro_f1"],
                "ROC-AUC": scores["roc_auc"],
            }
        )

baseline_table = pd.DataFrame(baseline_rows)

baseline_summary = (
    baseline_table.groupby("Baseline", sort=False)[["Accuracy", "Macro F1", "ROC-AUC"]]
    .mean()
    .round(4)
)

display(baseline_summary)

,Accuracy,Macro F1,ROC-AUC
Baseline,,,
Majority Class,0.6260,0.3849,0.5000
Always Beat,0.6260,0.3849,0.5000
Previous Outcome,0.6459,0.6217,0.6218
Historical Beat Rate,0.6638,0.6043,0.6725


#### Baseline Comparison

The Majority Class and Always Beat predictors produced identical results because earnings beats were the majority class in every training fold. Their mean accuracy was 0.6260, but their macro F1 was only 0.3849 because they failed to identify any meet-or-miss outcomes.

The Previous Outcome baseline achieved the highest macro F1 at 0.6217, compared with 0.6043 for Historical Beat Rate. However, Historical Beat Rate had higher accuracy and ROC-AUC.

I found that using previous earnings outcomes provided a substantial improvement in macro F1 over predicting the majority class. These results establish a useful benchmark for determining whether the machine learning models provide additional predictive value.